#서울시 내 녹지 -> 지도 시각화, shp 확장자 사용 

In [7]:
import geopandas as gpd
import numpy as np 
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.dates import date2num
import numpy as np 

from mpl_toolkits.basemap import Basemap
import matplotlib.cm as cm
import matplotlib as mpl
mpl.rcParams['font.family'] = 'Malgun Gothic'
mpl.rcParams['axes.unicode_minus'] = False


In [8]:
import geopandas as gpd
shp_path = r"UPIS_C_UQ153.shp"
# 인코딩 지정하여 불러오기
gdf = gpd.read_file(shp_path, encoding="cp949")
gdf.head()


DataSourceError: UPIS_C_UQ153.shp: No such file or directory

In [ ]:
green_codes = ['UQT200', 'UQT205', 'UQT210', 'UQT220', 'UQT230', 'UQT240']  # 예시: 공원 관련 코드
park_gdf = gdf[gdf['LCLAS_CL'].isin(green_codes)]
print(park_gdf.columns)

Index(['OBJECTID', 'PRESENT_SN', 'LCLAS_CL', 'MLSFC_CL', 'SCLAS_CL', 'ATRB_SE',
       'WTNNC_SN', 'NTFC_SN', 'DGM_NM', 'DGM_AR', 'DGM_LT', 'SIGNGU_SE',
       'DRAWING_NO', 'EXCUT_SE', 'CREATE_DAT', 'geometry'],
      dtype='object')


In [ ]:
area_by_green = park_gdf.groupby("SIGNGU_SE")["DGM_AR"].sum().reset_index()
area_by_green.columns = ["자치구코드", "green_area"]
area_by_green

,자치구코드,green_area
0,11000,1.934459e+08
1,11110,2.110992e+03
2,11140,3.907951e+04
3,11170,1.270371e+05
4,11200,5.877701e+04
5,11215,1.239778e+03
6,11230,2.339232e+06
7,11260,3.420455e+05
8,11290,5.459450e+04
9,11305,2.455805e+04


In [ ]:
import pandas as pd

# CSV 파일 경로
csv_path = r"C:\Users\LYJ\Desktop\CODE_HOME\PYTHON_HOME\25_Data_Anlysis\data\envior\서울시_자치구_중심점_2017.csv"

# 인코딩 주의: cp949 또는 euc-kr
gu_mapping_df = pd.read_csv(csv_path, encoding="cp949")
gu_mapping_df.head()


,코드,시도명,시군구명,X,Y
0,11110,서울특별시,종로구,126.977321,37.594917
1,11140,서울특별시,중구,126.995968,37.560144
2,11170,서울특별시,용산구,126.979907,37.531385
3,11200,서울특별시,성동구,127.041059,37.551030
4,11215,서울특별시,광진구,127.085744,37.546706


In [ ]:
gu_mapping_df = pd.read_csv(csv_path, encoding="cp949")
seoul_mapping = gu_mapping_df[gu_mapping_df["시도명"] == "서울특별시"]
seoul_mapping


,코드,시도명,시군구명,X,Y
0,11110,서울특별시,종로구,126.977321,37.594917
1,11140,서울특별시,중구,126.995968,37.560144
2,11170,서울특별시,용산구,126.979907,37.531385
3,11200,서울특별시,성동구,127.041059,37.551030
4,11215,서울특별시,광진구,127.085744,37.546706
5,11230,서울특별시,동대문구,127.054848,37.581957
6,11260,서울특별시,중랑구,127.092880,37.597803
7,11290,서울특별시,성북구,127.017579,37.605702
8,11305,서울특별시,강북구,127.011189,37.643474
9,11320,서울특별시,도봉구,127.032369,37.669102


In [ ]:
# 두 키 모두 정수형으로 맞추기
seoul_mapping["코드"] = seoul_mapping["코드"].astype(int)
area_by_green["자치구코드"] = area_by_green["자치구코드"].astype(int)

merged = pd.merge(seoul_mapping, area_by_green, left_on="코드", right_on="자치구코드")


In [ ]:
import folium

# 1. 서울 중심 좌표
center = [37.541, 126.986]

# 2. 사용할 타일 종류
tiles = ['cartodbpositron', 'Stamen Toner', 'OpenStreetMap']

# 3. 지도 생성
m = folium.Map(
    location=center,
    zoom_start=12,
    tiles=tiles[0]  # cartodbpositron 스타일
)

# 4. 지도 출력


In [ ]:
# 지도 생성
m = folium.Map(location=center, zoom_start=12, tiles=tiles[0])

# 자치구별 공원 면적 표시 (원 크기 축소)
for idx, row in merged.iterrows():
    folium.CircleMarker(
        location=[row['Y'], row['X']],
        radius=max(row['green_area'] / 180000, 6),  # 👈 radius 축소 + 최소값 설정
        color='green',
        fill=True,
        fill_color='green',
        fill_opacity=0.6,
        popup=f"{row['시군구명']} ({int(row['green_area'])}㎡)"
    ).add_to(m)

m


In [ ]:
green_table = merged[['시군구명', 'X', 'Y', 'green_area']]
green_table.columns = ['도시명', 'x', 'y', '녹지_면적']
green_table = green_table.sort_values(by='녹지_면적', ascending=False)

# 보기
display(green_table)

,도시명,x,y,녹지_면적
10,노원구,127.075035,37.652511,1.418485e+07
23,송파구,127.115295,37.505619,5.573413e+06
11,은평구,126.927023,37.619211,4.737024e+06
19,동작구,126.951641,37.498877,3.053799e+06
5,동대문구,127.054848,37.581957,2.339232e+06
16,구로구,126.856301,37.494405,6.684136e+05
6,중랑구,127.092880,37.597803,3.420455e+05
12,서대문구,126.939063,37.577785,2.695056e+05
18,영등포구,126.910169,37.522308,2.120104e+05
2,용산구,126.979907,37.531385,1.270371e+05


In [ ]:
stress_list = [
    ("강북구", 29.6),
    ("강서구", 28.8),
    ("마포구", 28.0),
    ("도봉구", 27.5),
    ("성북구", 27.0),
    ("중랑구", 26.8),
    ("은평구", 26.5),
    ("노원구", 26.2),
    ("동대문구", 25.9),
    ("광진구", 25.7),
    ("성동구", 25.4),
    ("용산구", 25.1),
    ("중구", 24.8),
    ("종로구", 24.5),
    ("서대문구", 24.2),
    ("양천구", 23.9),
    ("구로구", 23.6),
    ("금천구", 23.3),
    ("영등포구", 23.0),
    ("동작구", 22.7),
    ("관악구", 22.4),
    ("서초구", 22.1),
    ("강남구", 21.8),
    ("송파구", 21.5),
    ("강동구", 21.2)
]


In [ ]:
path = r"C:\Users\LYJ\Desktop\CODE_HOME\PYTHON_HOME\25_Data_Anlysis\data\envior\서울시 정신건강 통계간행물 목록.csv"
stress_df = gpd.read_file(shp_path, encoding="cp949")
stress_df.head()


,OBJECTID,PRESENT_SN,LCLAS_CL,MLSFC_CL,SCLAS_CL,ATRB_SE,WTNNC_SN,NTFC_SN,DGM_NM,DGM_AR,DGM_LT,SIGNGU_SE,DRAWING_NO,EXCUT_SE,CREATE_DAT,geometry
0,91486,11000UQ153PS201912154882,UQT500,UQT510,None,UQT510,11000URZ201001075014,11000NTC201001074399,공공공지,907.614757,623.489894,11000,None,EMA0009,2019-12-15,"POLYGON ((209069.592 457205.488, 209069.575 45..."
1,91487,11000UQ153PS201912154883,UQT500,UQT510,None,UQT510,11000URZ201405231774,11000NTC201405237206,공공공지,19030.159974,718.341159,11000,None,EMA0009,2019-12-15,"POLYGON ((196781.691 450030.401, 196792.642 45..."
2,91488,11230UQ153PS202004070001,UQT200,UQT210,None,UQT210,11230URZ202004010005,11230NTC202004010006,어린이공원,1272.736747,178.613500,11230,None,EMA0009,2020-04-07,"POLYGON ((202289.713 453093.575, 202286.237 45..."
3,91489,11230UQ153PS202004070002,UQT200,UQT205,None,UQT205,11230URZ202004010003,11230NTC202004010006,소공원,136.978577,45.836165,11230,None,EMA0009,2020-04-07,"POLYGON ((205087.476 452223.509, 205080.543 45..."
4,91490,11000UQ153PS202007220027,UQT200,UQT220,None,UQT220,11000URZ202007210096,11000NTC202007210001,근린공원,234463.479035,5721.092545,11000,22,EMA0009,2020-07-22,"MULTIPOLYGON (((202080.326 454721.854, 202078...."
